In [1]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import numpy as np
from keras.datasets import imdb

corpus = ["Hello", "Hello World", "Hello All"]


tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
sequences = []
for line in corpus:
    encoded = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(encoded)):
        sequence = encoded[:i+1]
        sequences.append(sequence)


max_length = max([len(seq) for seq in sequences])
sequences = np.array([np.pad(seq, (max_length - len(seq), 0), 'constant') for seq in sequences])
X, y = sequences[:,:-1], sequences[:,-1]
y = to_categorical(y, num_classes=len(tokenizer.word_index)+1)

vocab_size = len(tokenizer.word_index) + 1


model = Sequential()
#model.add(Embedding(input_dim=len(tokenizer.word_index)+1, output_dim=10, input_length=max_length-1))
model.add(Embedding(input_dim=vocab_size, output_dim=10))

model.add(LSTM(50))
model.add(Dense(len(tokenizer.word_index)+1, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=2)


Epoch 1/100
1/1 - 2s - 2s/step - accuracy: 0.5000 - loss: 1.3864
Epoch 2/100
1/1 - 0s - 39ms/step - accuracy: 0.5000 - loss: 1.3840
Epoch 3/100
1/1 - 0s - 46ms/step - accuracy: 0.5000 - loss: 1.3816
Epoch 4/100
1/1 - 0s - 35ms/step - accuracy: 0.5000 - loss: 1.3791
Epoch 5/100
1/1 - 0s - 48ms/step - accuracy: 0.5000 - loss: 1.3766
Epoch 6/100
1/1 - 0s - 35ms/step - accuracy: 0.5000 - loss: 1.3742
Epoch 7/100
1/1 - 0s - 48ms/step - accuracy: 0.5000 - loss: 1.3717
Epoch 8/100
1/1 - 0s - 35ms/step - accuracy: 0.5000 - loss: 1.3691
Epoch 9/100
1/1 - 0s - 34ms/step - accuracy: 0.5000 - loss: 1.3666
Epoch 10/100
1/1 - 0s - 34ms/step - accuracy: 0.5000 - loss: 1.3639
Epoch 11/100
1/1 - 0s - 33ms/step - accuracy: 0.5000 - loss: 1.3613
Epoch 12/100
1/1 - 0s - 48ms/step - accuracy: 0.5000 - loss: 1.3586
Epoch 13/100
1/1 - 0s - 35ms/step - accuracy: 0.5000 - loss: 1.3558
Epoch 14/100
1/1 - 0s - 50ms/step - accuracy: 0.5000 - loss: 1.3530
Epoch 15/100
1/1 - 0s - 32ms/step - accuracy: 0.5000 - loss

In [3]:
def predict_next_word(model, tokenizer, text_input, max_len):
    encoded = tokenizer.texts_to_sequences([text_input])[0]
    padded = np.pad(encoded, (max_len - len(encoded), 0), 'constant')
    padded = padded.reshape(1, -1)
    pred_index = np.argmax(model.predict(padded), axis=-1)[0]
    
    for word, index in tokenizer.word_index.items():
        if index == pred_index:
            return word
    return None
    
input_text = "Hey"
next_word = predict_next_word(model, tokenizer, input_text, max_length-1)
print(f"Input: '{input_text}' → Predicted next word: '{next_word}'")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
Input: 'Hey' → Predicted next word: 'all'
